# Export the SudhishNeel Clash Royale detector to ONNX

**Runtime → Run all.** No GPU needed. Takes about 2 minutes. Ignore any 'restart runtime' prompt: this cell installs and exports in one go.

When it finishes your browser downloads `sudhishneel_v8_onnx.zip`. If it doesn't: click the **folder icon** on the left, right-click `sudhishneel_v8_onnx.zip` → **Download**.

Unzip into `CR-Pro-Coach/pretrained_weights/sudhishneel_v8/` (you should get `v8_best.onnx` and `names.json`).

Every step prints a numbered line (1)–(4). If one is missing, copy the red error text back to Claude.


In [ ]:
# CR Bot: export SudhishNeel v8 detector to ONNX (one cell, fixed paths). Run with Runtime -> Run all.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ultralytics", "onnx", "onnxslim", "onnxruntime", "huggingface_hub"], check=True)

import os, json, hashlib, shutil, zipfile, pickletools, io, glob
os.makedirs("/content/work", exist_ok=True)
os.chdir("/content/work")

# 1) download the exact file that was checked on the Mac
from huggingface_hub import hf_hub_download
pt = hf_hub_download("SudhishNeel/clash-royale-detector", "v8/best.pt", local_dir="/content/work/hf")
sha = hashlib.sha256(open(pt, "rb").read()).hexdigest()
print("1) downloaded", pt, "sha256", sha)
assert sha == "af0046fcd640965f0527e798615824928bf6e7360fe1d17b09e51b0492391cfa", "STOP: file changed since it was checked"

# 2) safety scan: list what the pickle refers to, without loading it
refs, z = set(), zipfile.ZipFile(pt)
for n in [n for n in z.namelist() if n.endswith(".pkl")]:
    strings = []
    for op, arg, _ in pickletools.genops(io.BytesIO(z.read(n))):
        if isinstance(arg, str):
            strings.append(arg)
        if op.name == "GLOBAL":
            refs.add(arg.split(" ")[0].split(".")[0])
        elif op.name == "STACK_GLOBAL":
            refs.add(strings[-2].split(".")[0])
print("2) pickle refers to:", sorted(refs))
assert refs <= {"torch", "ultralytics", "collections", "__builtin__", "builtins"}, "STOP: unexpected modules"

# 3) export
from ultralytics import YOLO
model = YOLO(pt)
names = [model.names[i] for i in range(len(model.names))]
onnx_path = model.export(format="onnx", imgsz=800, dynamic=False, simplify=True, opset=17)
print("3) exported to", onnx_path, "| classes:", len(names))

# 4) package
out = "/content/out"
os.makedirs(out, exist_ok=True)
shutil.copy(onnx_path, f"{out}/v8_best.onnx")
json.dump(names, open(f"{out}/names.json", "w"))
zip_path = shutil.make_archive("/content/sudhishneel_v8_onnx", "zip", out)
for f in sorted(os.listdir(out)):
    print("4)", f"{out}/{f}", round(os.path.getsize(f"{out}/{f}") / 1e6, 2), "MB")
print("4) zip:", zip_path)

# 5) download (if the browser blocks it: folder icon on the left -> right-click the zip -> Download)
from google.colab import files
files.download(zip_path)